## Setup paths

In [ ]:
from pathlib import Path
import os

# Project root: assumes the notebook is run from the notebooks/ folder
PROJECT_ROOT = Path.cwd().resolve().parent

# Main project folders
src_dir = PROJECT_ROOT / "src"
wrk_dir = str(src_dir / "dependencies")
sub_dir = str(PROJECT_ROOT / "data" / "subjects")

# Define folder name of subject
subject_folder = "TBI-generic-50perc-avg-male"

# Define output folder name
output_name = "output"

# Input image names expected inside:
# data/subjects/<subject_folder>/img/fs_seg/
T1 = "T1.nii.gz"
T2 = ""

## Image customisation

In [ ]:
# Minimum intensity value in T1 of image values where noise is lower
# For more details see: https://fsl.fmrib.ox.ac.uk/fsl/fslwiki/SUSAN
threshold_filter = 10

# Percent of skull from bottom to exclude to keep pricavy of individual, from 
# inferior to superior. 30% is normally adequate to remove most
# of the facial features but leave behind enough for good visuals
skull_privacy_percentage = 0

## Checking folders

In [ ]:
%%bash -s "$sub_dir" "$subject_folder"

echo "Subject root: $1"
SUB_PATH="$1/$2"

if [ ! -d "$SUB_PATH" ]; then
    echo "Subject $2 not found in $1"
    exit 1
fi

echo "Setting up folders in $SUB_PATH"
cd "$SUB_PATH" || exit 1

mkdir -p tmp
mkdir -p img/fast
mkdir -p img/bet
mkdir -p output

echo "Folders ready in $SUB_PATH"

## Rigid registration and Image Alignment

In [ ]:
%%bash -s "$sub_dir" "$subject_folder"

set -e

SUB_PATH="$1/$2"
IMG_DIR="${SUB_PATH}/img"
FSSEG_DIR="${IMG_DIR}/fs_seg"
TMP_DIR="${SUB_PATH}/tmp"
FAST_DIR="${IMG_DIR}/fast"

mkdir -p "$FAST_DIR" "$TMP_DIR"

# Standard MNI brain template from local FSL install
MNI_REF="${FSLDIR}/data/standard/MNI152_T1_1mm_brain.nii.gz"

echo "Using subject folder: $SUB_PATH"
echo "Using fs_seg folder: $FSSEG_DIR"

# Check required inputs
if [ ! -f "${FSSEG_DIR}/brain.nii.gz" ]; then
    echo "Missing file: ${FSSEG_DIR}/brain.nii.gz"
    exit 1
fi

if [ ! -f "${FSSEG_DIR}/T1.nii.gz" ]; then
    echo "Missing file: ${FSSEG_DIR}/T1.nii.gz"
    exit 1
fi

if [ ! -f "${FSSEG_DIR}/aseg.nii.gz" ]; then
    echo "Missing file: ${FSSEG_DIR}/aseg.nii.gz"
    exit 1
fi

# Copy inputs into tmp so downstream steps keep the same filenames
echo "Step 1: Copying input NIfTI files..."
cp "${FSSEG_DIR}/brain.nii.gz" "${TMP_DIR}/brain_raw.nii.gz"
cp "${FSSEG_DIR}/T1.nii.gz" "${TMP_DIR}/T1_raw.nii.gz"
cp "${FSSEG_DIR}/aseg.nii.gz" "${TMP_DIR}/aseg_raw.nii.gz"

# Rigid registration using skull-stripped brain
echo "Step 2: Calculating rigid registration (brain -> MNI)..."
flirt -in "${TMP_DIR}/brain_raw.nii.gz" \
      -ref "${MNI_REF}" \
      -out "${FAST_DIR}/brain_std.nii.gz" \
      -omat "${TMP_DIR}/brain_to_mni.mat" \
      -dof 6 \
      -searchrx -90 90 -searchry -90 90 -searchrz -90 90

# Apply transform to T1
echo "Step 3: Applying transform to T1..."
flirt -in "${TMP_DIR}/T1_raw.nii.gz" \
      -ref "${MNI_REF}" \
      -out "${TMP_DIR}/T1_std.nii.gz" \
      -applyxfm -init "${TMP_DIR}/brain_to_mni.mat" \
      -interp trilinear

# Apply transform to aseg using nearest-neighbour interpolation
echo "Step 4: Applying transform to aseg..."
flirt -in "${TMP_DIR}/aseg_raw.nii.gz" \
      -ref "${MNI_REF}" \
      -out "${TMP_DIR}/aseg_std.nii.gz" \
      -applyxfm -init "${TMP_DIR}/brain_to_mni.mat" \
      -interp nearestneighbour

echo "Rigid registration complete."

##  Running _fast_

In [ ]:
%%bash -s "$sub_dir" "$subject_folder"

set -e

SUB_PATH="$1/$2"
cd "$SUB_PATH" || exit 1
echo "Moved to: $SUB_PATH"

# Optional alignment sanity check
flirt -in tmp/T1_std.nii.gz \
      -ref tmp/aseg_std.nii.gz \
      -omat tmp/T1_to_aseg.mat \
      -nosearch -dof 6

# Run FAST on the MNI-registered brain
cd img/fast || exit 1
echo "Running FAST on MNI-registered data..."
fast brain_std.nii.gz
cd ../../

## Running _bet_

In [ ]:
%%bash -s "$sub_dir" "$subject_folder"

set -e

SUB_PATH="$1/$2"
cd "$SUB_PATH" || exit 1
echo "Moved to: $SUB_PATH"

# Run BET on MNI-registered T1
bet tmp/T1_std img/bet/T1_bet -s -f 0.35 -g -0.05 -e -R

# Create identity matrix if it doesn't exist
if [ ! -f img/bet/identity.mat ]; then
  echo -e "1 0 0 0\n0 1 0 0\n0 0 1 0\n0 0 0 1" > img/bet/identity.mat
fi

# Run BETSURF
betsurf --t1only -m tmp/T1_std img/bet/T1_bet_mesh.vtk img/bet/identity.mat img/bet/betsurf

In [ ]:
%%bash -s "$sub_dir" "$subject_folder"

set -e

SUB_PATH="$1/$2"
cd "$SUB_PATH" || exit 1
echo "Moved to: $SUB_PATH"

fslmaths tmp/T1_std.nii.gz -thr 39 -bin tmp/wholehead_mask.nii.gz
fslmaths tmp/wholehead_mask.nii.gz -add img/bet/betsurf_outskin_mask.nii.gz -bin img/bet/betsurf_outskin_mask.nii.gz

## Creating brain geometry (pre_model)

In [ ]:
%%bash -s "$sub_dir" "$subject_folder" "$threshold_filter" "$skull_privacy_percentage"

set -e

SUB_PATH="$1/$2"
THRESH="$3"
PRIVACY="$4"

cd "$SUB_PATH" || exit 1
echo "Moved to: $SUB_PATH"

echo "Processing brain segmentation"

# Extract FAST results for CSF, GM, WM
fslmaths img/fast/brain_std_seg.nii.gz -thr 0.5 -uthr 1.5 -bin tmp/csf_fast.nii.gz
fslmaths img/fast/brain_std_seg.nii.gz -thr 1.5 -uthr 2.5 -bin tmp/gm_fast.nii.gz
fslmaths img/fast/brain_std_seg.nii.gz -thr 2.5 -uthr 3.5 -bin tmp/wm_fast.nii.gz

# Find missing brain matter
fslmaths tmp/aseg_std.nii.gz -bin tmp/aseg_std_bin.nii.gz
fslmaths tmp/gm_fast.nii.gz -bin tmp/gm_fast_bin.nii.gz
fslmaths tmp/wm_fast.nii.gz -bin tmp/wm_fast_bin.nii.gz
fslmaths tmp/wm_fast_bin.nii.gz -add tmp/gm_fast_bin.nii.gz tmp/brain_fast.nii.gz
fslmaths tmp/brain_fast.nii.gz -sub tmp/aseg_std_bin.nii.gz -thr 1 -ero -dilM -dilM -ero -bin -mul 300 tmp/bm.nii.gz

# Add missing brain matter to aseg
fslmaths tmp/bm.nii.gz -bin -mul 1000 tmp/bm_bin.nii.gz
fslmaths tmp/aseg_std.nii.gz -sub tmp/bm_bin.nii.gz -thr 1 tmp/aseg_std.nii.gz
fslmaths tmp/bm.nii.gz -add tmp/aseg_std.nii.gz tmp/aseg_bm.nii.gz

echo "Adding CSF to segmentation"

# Remove ventricles from FAST CSF segmentation
fslmaths tmp/csf_fast.nii.gz -dilM -ero tmp/csf_fast.nii.gz
fslmaths tmp/aseg_bm.nii.gz -thr 42.5 -uthr 43.5 tmp/ventricleR.nii.gz
fslmaths tmp/aseg_bm.nii.gz -thr 3.5 -uthr 4.5 tmp/ventricleL.nii.gz
fslmaths tmp/ventricleR.nii.gz -add tmp/ventricleL.nii.gz tmp/ventricles_both.nii.gz
fslmaths tmp/ventricles_both.nii.gz -bin tmp/ventricles_both.nii.gz
fslmaths tmp/csf_fast.nii.gz -sub tmp/ventricles_both.nii.gz tmp/csf_no_vent.nii.gz
fslmaths tmp/csf_no_vent.nii.gz -thr 0.9 -uthr 1.1 tmp/csf_no_vent.nii.gz

# Add CSF to brain
fslmaths tmp/aseg_bm.nii.gz -bin tmp/aseg_bm_bin.nii.gz
fslmaths tmp/csf_no_vent.nii.gz -bin -dilM -ero tmp/csf_no_vent_bin.nii.gz
fslmaths tmp/csf_no_vent_bin.nii.gz -sub tmp/aseg_bm_bin.nii.gz -thr 1 -mul 256 tmp/csf.nii.gz
fslmaths tmp/aseg_bm.nii.gz -add tmp/csf.nii.gz tmp/aseg_bm_csf.nii.gz

# Remove holes
fslmaths tmp/aseg_bm_csf.nii.gz -bin tmp/aseg_bm_csf_bin.nii.gz
fslmaths tmp/aseg_bm_csf_bin.nii.gz -dilM -dilM -ero -ero tmp/aseg_bm_csf_bin_mask.nii.gz
fslmaths tmp/aseg_bm_csf_bin_mask.nii.gz -sub tmp/aseg_bm_csf_bin.nii.gz tmp/csf_missing.nii.gz
fslmaths tmp/csf_missing.nii.gz -mul 256 tmp/csf_missing.nii.gz
fslmaths tmp/csf_missing.nii.gz -add tmp/aseg_bm_csf.nii.gz tmp/aseg_bm_csf.nii.gz

# Skull surrounding brain
fslmaths tmp/aseg_bm_csf.nii.gz -bin -dilM -dilM -ero -ero tmp/aseg_bm_csf_bin.nii.gz
fslmaths tmp/aseg_bm_csf_bin.nii.gz -ero -dilM -dilM tmp/aseg_bm_csf_bin_dil.nii.gz
fslmaths tmp/aseg_bm_csf_bin_dil.nii.gz -sub tmp/aseg_bm_csf_bin.nii.gz tmp/skull_outline.nii.gz

# Visual part of skull
fslmaths img/bet/betsurf_outskull_mask.nii.gz -sub tmp/aseg_bm_csf_bin.nii.gz tmp/T1_std_roi.nii.gz

# Combine visual and surrounding skull
fslmaths tmp/T1_std_roi.nii.gz -add tmp/skull_outline.nii.gz tmp/skull_mask.nii.gz
fslmaths tmp/skull_mask.nii.gz -bin tmp/skull_mask.nii.gz

# Ensure no overlap of skull with CSF and brain
fslmaths tmp/aseg_bm_csf.nii.gz -bin tmp/aseg_bm_csf_bin.nii.gz
fslmaths tmp/skull_mask.nii.gz -sub tmp/aseg_bm_csf_bin.nii.gz -thr 1 tmp/skull_mask.nii.gz
fslmaths tmp/skull_mask.nii.gz -mul 257 tmp/skull_mask.nii.gz

# Combine skull, CSF, GM, WM, and subcortical structures
fslmaths tmp/aseg_bm_csf.nii.gz -add tmp/skull_mask.nii.gz tmp/aseg_bm_csf_skull.nii.gz

# Create skin mask
fslmaths tmp/aseg_bm_csf_skull.nii.gz -bin tmp/aseg_bm_csf_skull_bin.nii.gz
fslmaths img/bet/betsurf_outskin_mask.nii.gz -sub tmp/aseg_bm_csf_skull_bin.nii.gz tmp/skin_mask.nii.gz

# Flatten bottom and anonymise
echo "$(fslhd tmp/T1_std.nii.gz)" > tmp/tmp.txt
dim3=$(awk '{for (I=1;I<=NF;I++) if ($I == "dim3") {print $(I+1)};}' "tmp/tmp.txt")
dim3_min=$(($dim3 * $PRIVACY / 100))
fslmaths tmp/skin_mask.nii.gz -roi 0 -1 0 -1 $dim3_min -1 0 -1 -bin tmp/skin_mask_anon.nii.gz

# Add skin to segmentation
fslmaths tmp/skin_mask_anon.nii.gz -mul 260 tmp/skin_mask_anon.nii.gz
fslmaths tmp/skin_mask_anon.nii.gz -add tmp/aseg_bm_csf_skull.nii.gz tmp/aseg_bm_csf_skull_skin.nii.gz
cp tmp/aseg_bm_csf_skull_skin.nii.gz pre_model.nii.gz

echo "Created pre_model.nii.gz"